In [ ]:
pip install neuralhydrology


In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from neuralhydrology.evaluation import metrics, get_tester
from neuralhydrology.nh_run import start_run, eval_run
from neuralhydrology.utils.config import Config

In [ ]:
#baseline test period
run_dir = Path("/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/runs/no_soil_1908_132818")   # you'll find this path in the output of the training above.
run_config = Config(Path(run_dir/"config.yml"))
tester = get_tester(cfg=Config(run_dir / "config.yml"), run_dir=run_dir, period="test", init_model=True)
results = tester.evaluate(save_results=True, metrics=run_config.metrics)

results.keys()

In [ ]:
# extract observations and simulations
daily_qobs = results["Washita"]["1D"]["xr"]["QObs_obs"]
daily_qsim_baseline = results["Washita"]["1D"]["xr"]["QObs_sim"]

fig, ax = plt.subplots(figsize=(16,10))
ax.plot(daily_qobs["date"], daily_qobs, label="Observed")
ax.plot(daily_qsim_baseline["date"], daily_qsim_baseline, label="Simulated")
ax.legend()
ax.set_ylabel("Discharge (cubic feet/sec)")
ax.set_title("Time Series for test period")

# Calculate some metrics
values = metrics.calculate_all_metrics(daily_qobs.isel(time_step=-1), daily_qsim_baseline.isel(time_step=-1))
print("Daily metrics:")
for key, val in values.items():
    print(f"  {key}: {val:.3f}")

In [ ]:
#soil moisture test period
run_dir = Path("/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/runs/with_soil_1908_151843")   # you'll find this path in the output of the training above.
run_config = Config(Path(run_dir/"config.yml"))
# create a tester instance and start evaluation
tester = get_tester(cfg=Config(run_dir / "config.yml"), run_dir=run_dir, period="test", init_model=True)
results = tester.evaluate(save_results=True, metrics=run_config.metrics)

results.keys()

In [ ]:
# extract observations and simulations
daily_qobs = results["Washita"]["1D"]["xr"]["QObs_obs"]
daily_qsim = results["Washita"]["1D"]["xr"]["QObs_sim"]

fig, ax = plt.subplots(figsize=(16,10))
ax.plot(daily_qobs["date"], daily_qobs, label="Observed", color = "red")
ax.plot(daily_qsim["date"], daily_qsim, label="Simulated_withSM")
ax.plot(daily_qsim_baseline["date"], daily_qsim_baseline, label="Simulated_baseline", color = "pink")
ax.legend()
ax.set_ylabel("Discharge (cubic feet/sec)")
ax.set_title("Time Series for Test period")

# Calculate some metrics
values = metrics.calculate_all_metrics(daily_qobs.isel(time_step=-1), daily_qsim.isel(time_step=-1))
print("Daily metrics:")
for key, val in values.items():
    print(f"  {key}: {val:.3f}")

In [ ]:
print(daily_qobs)
print(daily_qsim)
print(daily_qsim_baseline)

In [ ]:
# Convert xarray Dataset to pandas DataFrame
import xarray as xr

# Merge the datasets along a new 'scenario' dimension
merged_ds = xr.concat([daily_qobs, daily_qsim, daily_qsim_baseline], dim="scenario")

# Assign scenario labels
merged_ds = merged_ds.assign_coords(scenario=["obs", "sm", "baseline"])

# Convert to DataFrame
df = merged_ds.to_dataframe(name="value").reset_index()

# Pivot to get separate columns for 'obs', 'sm', 'baseline'
df_pivot = df.pivot(index="date", columns="scenario", values="value")

# Save as CSV
df_pivot.to_csv("/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/sequence90_test/seed10.csv")

print("CSV file saved successfully!")



In [ ]:
#averaging for 10 seeds for each sequence length
import pandas as pd
import glob

# Path to your CSV files (modify as needed)
file_paths = glob.glob("/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/sequence90_test/seed*.csv")

# Read and merge all CSV files
df_list = [pd.read_csv(file) for file in file_paths]

# Merge on 'date' using outer join to include all dates
merged_df = pd.concat(df_list).groupby("date", as_index=False).mean()

# Save the final merged dataset
merged_df.to_csv("/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/sequence90_test/merged_mean_seq90.csv", index=False)

print("Merged CSV with mean values saved successfully!")


In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt

df = pd.read_csv("/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/sequence90_test/merged_mean_seq90.csv",index_col= False)
df["date"] = pd.to_datetime(df["date"])

plt.figure(figsize=(12, 5))
plt.plot(df["date"],df["baseline"], label = "baseline", color = "red", linewidth = 0.5)
plt.plot(df["date"],df["sm"], label = "with_SM", color ="green", linewidth = 0.5)
plt.plot(df["date"],df["obs"], label = "observed", color = "black", linewidth = 0.5)
plt.legend()
plt.xlabel("date")

plt.ylabel ("Discharge(cubic feet/sec)")


In [ ]:
#try plotting the particular years
#2018

import matplotlib.pyplot as plt

import matplotlib.dates as mdates
import pandas as pd
import numpy as np
import pandas as pd
import glob

df = pd.read_csv("/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/sequence30_test/merged_mean_seq30.csv",index_col= False)
df["date"] = pd.to_datetime(df["date"])

# Convert cfs to m3/s
conversion = 0.0283168

df["obs"] = df["obs"] * conversion
df["baseline"] = df["baseline"] * conversion
df["sm"] = df["sm"] * conversion

# Filter for the year 2015
df_2018 = df[(df["date"] >= "2018-01-01") & (df["date"] <= "2018-12-31")]

# Define NSE function
def nse(simulated, observed):
    return 1 - np.sum((observed - simulated) ** 2) / np.sum((observed - np.mean(observed)) ** 2)

# Calculate NSE for baseline and sm
nse_baseline_2018 = nse(df_2018['baseline'], df_2018['obs'])
nse_sm_2018 = nse(df_2018['sm'], df_2018['obs'])

# Print results
print(f"NSE for Baseline in 2018: {nse_baseline_2018:.4f}")
print(f"NSE for SM in 2018: {nse_sm_2018:.4f}")
# Create plot
plt.figure(figsize=(12, 6))

plt.plot(df_2018["date"],df_2018["baseline"], label = "baseline", color = "#E69F00", linewidth = 0.7)
plt.plot(df_2018["date"],df_2018["sm"], label = "with soil moisture", color = "#2ca25f", linewidth = 0.7)
plt.plot(df_2018["date"],df_2018["obs"], label = "observed", color = "black", linewidth = 0.7)
plt.legend(fontsize = 19, loc = "upper center", edgecolor='gray')
plt.xlabel("Date", fontsize = 24)
plt.ylabel ("Discharge(m$^3$/sec)", fontsize = 24)


plt.xticks(rotation=45, fontsize = 20)
plt.yticks(fontsize = 20)
plt.grid(True)

# Add NSE values as text in the plot
plt.text(df_2018["date"].min(), 
         df_2018["baseline"].max() * 0.99,  # adjust vertical position
         f"NSE (Baseline): {nse_baseline_2018:.2f}\nNSE (SM): {nse_sm_2018:.2f}",
         fontsize=19, 
         verticalalignment='top',
         bbox=dict(facecolor='white', alpha=0.7, edgecolor='gray'))

plt.tight_layout()
plt.savefig("hydrograph.png", dpi=600, bbox_inches='tight')
plt.show()







In [ ]:
#2020
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Read data
df = pd.read_csv(
    "/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/sequence90_test/merged_mean_seq90.csv",
    index_col=False
)

df["date"] = pd.to_datetime(df["date"])

# Convert cfs to m3/s
conversion = 0.0283168

df["obs"] = df["obs"] * conversion
df["baseline"] = df["baseline"] * conversion
df["sm"] = df["sm"] * conversion

# Filter desired year
df_2020 = df[
    (df["date"] >= "2020-01-01") &
    (df["date"] <= "2020-12-31")
]

# NSE function
def nse(simulated, observed):
    return 1 - np.sum((observed - simulated) ** 2) / np.sum(
        (observed - np.mean(observed)) ** 2
    )

# Calculate NSE
nse_baseline_2020 = nse(df_2020["baseline"], df_2020["obs"])
nse_sm_2020 = nse(df_2020["sm"], df_2020["obs"])

# ------------------------
# Plot
# ------------------------
plt.figure(figsize=(10, 5))

plt.plot(
    df_2020["date"],
    df_2020["baseline"],
    label="Baseline",
    color="#E69F00",
    linewidth=1.0
)

plt.plot(
    df_2020["date"],
    df_2020["sm"],
    label="With Soil Moisture",
    color="#2ca25f",
    linewidth=1.0
)

plt.plot(
    df_2020["date"],
    df_2020["obs"],
    label="Observed",
    color="black",
    linewidth=1.0
)

# Labels
plt.xlabel("Date", fontsize=16)
plt.ylabel("Discharge (m$^3$/s)", fontsize=14)

# Ticks
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)

# Legend
plt.legend(
    fontsize=11,
    loc="upper right",
    edgecolor="gray"
)

# Grid
plt.grid(True, alpha=0.3)

# NSE annotation
plt.text(
    0.02, 0.96,
    f"NSE (Baseline): {nse_baseline_2020:.2f}\nNSE (SM): {nse_sm_2020:.2f}",
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment="top",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray")
)

plt.tight_layout()

# Save figure
plt.savefig(
    "hydrograph_2020.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# -----------------------------
# NSE function
# -----------------------------
def nse(simulated, observed):
    return 1 - np.sum((observed - simulated) ** 2) / np.sum((observed - np.mean(observed)) ** 2)

# -----------------------------
# Load both datasets
# -----------------------------
seq30 = pd.read_csv("/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/sequence30_test/merged_mean_seq30.csv")
seq90 = pd.read_csv("/Users/gautam/Research/codesforresearch/percentsaturation/after first review/Voronoi/sequence90_test/merged_mean_seq90.csv")

seq30["date"] = pd.to_datetime(seq30["date"])
seq90["date"] = pd.to_datetime(seq90["date"])

# -----------------------------
# Filter for years
# -----------------------------
def filter_year(df, year):
    return df[(df["date"] >= f"{year}-01-01") & (df["date"] <= f"{year}-12-31")]

# Convert cfs to m3/s
conversion = 0.0283168

for df in [seq30, seq90]:
    df["obs"] = df["obs"] * conversion
    df["baseline"] = df["baseline"] * conversion
    df["sm"] = df["sm"] * conversion

years = [2019, 2023]  # wettest and driest
seq30_2019 = filter_year(seq30, 2019)
seq30_2023 = filter_year(seq30, 2023)
seq90_2019 = filter_year(seq90, 2019)
seq90_2023 = filter_year(seq90, 2023)

# -----------------------------
# Make subplots (2x2)
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False)
ax = axes.ravel()

datasets = [
    (seq30_2019, "(a) Seq length 30", "Wettest Year (2019)"),
    (seq30_2023, "(b) Seq length 30", "Driest Year (2023)"),
    (seq90_2019, "(c) Seq length 90", "Wettest Year (2019)"),
    (seq90_2023, "(d) Seq length 90", "Driest Year (2023)")
]

colors = {
    "baseline": "#E69F00",
    "sm": "#2ca25f",
    "obs": "black"
}

# -----------------------------
# Create all 4 subplots
# -----------------------------
for i, (df, seq_label, year_label) in enumerate(datasets):

    # Compute NSE
    nse_base = nse(df["baseline"], df["obs"])
    nse_sm = nse(df["sm"], df["obs"])

    ax[i].plot(df["date"], df["obs"], label="Observed", color=colors["obs"], linewidth=0.7)
    ax[i].plot(df["date"], df["baseline"], label="Baseline", color=colors["baseline"], linewidth=0.7)
    ax[i].plot(df["date"], df["sm"], label="With Soil Moisture", color=colors["sm"], linewidth=0.7)

    ax[i].set_title(f"{seq_label} , {year_label}", fontsize=14)
    ax[i].grid(True)

    # Tick label font size (x and y)
    ax[i].tick_params(axis="both", labelsize=12)

    # Add NSE text box
    ax[i].text(
        0.01, 0.99,
        f"NSE (Baseline): {nse_base:.2f}\nNSE (SM(25)): {nse_sm:.2f}",
        transform=ax[i].transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='gray')
    )

# -----------------------------
# Formatting
# -----------------------------
fig.supxlabel("Date", fontsize=16)
fig.supylabel("Discharge (m$^3$/sec)", fontsize=14)

# Place one legend for all subplots
handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("combined_subplots.png", dpi=600)
plt.show()
